# Adatok felderítése, pipeline kialakítása

1: SZÜKSÉGES CSOMAGOK IMPORTÁLÁSA

In [29]:
import xarray as xr
from pathlib import Path
import numpy as np
import pandas as pd

kelvin = 273.15

2: DATA EXPLORATORY

In [30]:
files = sorted(Path(r"tigge_data").glob("*.grib"))
print(f"- Összesen {len(files)} fájlt tudtunk letölteni")

# kiválasztunk egy mintát
ds = xr.open_dataset(files[0], engine="cfgrib")

print()
print("- Nézzünk bele egy grib fájlba:")
print(ds)

- Összesen 158 fájlt tudtunk letölteni

- Nézzünk bele egy grib fájlba:
<xarray.Dataset> Size: 195kB
Dimensions:            (time: 31, step: 4, values: 97)
Coordinates:
  * time               (time) datetime64[ns] 248B 2006-10-01 ... 2006-10-31
  * step               (step) timedelta64[ns] 32B 06:00:00 ... 1 days 00:00:00
    valid_time         (time, step) datetime64[ns] 992B ...
    latitude           (values) float64 776B ...
    longitude          (values) float64 776B ...
    number             int64 8B ...
    heightAboveGround  float64 8B ...
    surface            float64 8B ...
Dimensions without coordinates: values
Data variables:
    mx2t6              (time, step, values) float32 48kB ...
    mn2t6              (time, step, values) float32 48kB ...
    t2m                (time, step, values) float32 48kB ...
    tp                 (time, step, values) float32 48kB ...
Attributes:
    GRIB_edition:            2
    GRIB_centre:             ecmf
    GRIB_centreDescription:  E

In [31]:
cities = {
    "Budapest": (47.4979, 19.0402),
    "Debrecen": (47.5316, 21.6273),
    "Pécs":     (46.0727, 18.2323),
    "Szeged":   (46.2530, 20.1414),
    "Szombathely": (47.23,16.62)
}

In [32]:
print(ds.latitude.values)

[48.763994   48.763994   48.763994   48.763994   48.763994   48.763994
 48.763994   48.763994   48.763994   48.763994   48.763994   48.763994
 48.3145566  48.3145566  48.3145566  48.3145566  48.3145566  48.3145566
 48.3145566  48.3145566  48.3145566  48.3145566  48.3145566  48.3145566
 47.86511918 47.86511918 47.86511918 47.86511918 47.86511918 47.86511918
 47.86511918 47.86511918 47.86511918 47.86511918 47.86511918 47.86511918
 47.41568175 47.41568175 47.41568175 47.41568175 47.41568175 47.41568175
 47.41568175 47.41568175 47.41568175 47.41568175 47.41568175 47.41568175
 46.96624431 46.96624431 46.96624431 46.96624431 46.96624431 46.96624431
 46.96624431 46.96624431 46.96624431 46.96624431 46.96624431 46.96624431
 46.51680685 46.51680685 46.51680685 46.51680685 46.51680685 46.51680685
 46.51680685 46.51680685 46.51680685 46.51680685 46.51680685 46.51680685
 46.06736938 46.06736938 46.06736938 46.06736938 46.06736938 46.06736938
 46.06736938 46.06736938 46.06736938 46.06736938 46.06736

In [33]:
def nearest_point(ds, cel_latitude, cel_longitude):
    lat = ds.latitude.values
    lon = ds.longitude.values
    distance = (lat - cel_latitude)**2 + (lon - cel_longitude)**2
    idx = np.argmin(distance)
    return ds.isel(values=idx)

In [34]:
print(nearest_point(ds,cities["Budapest"][0],cities["Budapest"][1]))

<xarray.Dataset> Size: 3kB
Dimensions:            (time: 31, step: 4)
Coordinates:
  * time               (time) datetime64[ns] 248B 2006-10-01 ... 2006-10-31
  * step               (step) timedelta64[ns] 32B 06:00:00 ... 1 days 00:00:00
    valid_time         (time, step) datetime64[ns] 992B ...
    number             int64 8B ...
    heightAboveGround  float64 8B ...
    latitude           float64 8B 47.42
    longitude          float64 8B 18.75
    surface            float64 8B ...
Data variables:
    mx2t6              (time, step) float32 496B ...
    mn2t6              (time, step) float32 496B ...
    t2m                (time, step) float32 496B ...
    tp                 (time, step) float32 496B ...
Attributes:
    GRIB_edition:            2
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Ra

In [35]:
print(ds.step.values)

[21600000000000 43200000000000 64800000000000 86400000000000]


3: EGY HÓNAP DATASET-JÉNEK ELŐÁLLÍTÁSA

In [36]:
def daily_aggregation(pont_ds):
    """pont_ds: egy adott város rácspontjára kivágott Dataset (time, step dimenziókkal)"""
    
    daily_max  = pont_ds["mx2t6"].max(dim="step")
    daily_min  = pont_ds["mn2t6"].min(dim="step")
    daily_mean_maxmin = (daily_max + daily_min) / 2
    daily_mean_t2m    = pont_ds["t2m"].mean(dim="step")
    
    # csapadék: kumulatív, a 24h (utolsó) lépés adja a teljes napi összeget
    daily_precipitation  = pont_ds["tp"].isel(step=-1)

    
    df = pd.DataFrame({
        "datum": pont_ds["time"].values,
        "tmax": daily_max.values,
        "tmin": daily_min.values,
        "tkozep_maxmin": daily_mean_maxmin.values,
        "tkozep_t2m": daily_mean_t2m.values,
        "csapadek_mm": daily_precipitation.values,
    })
    return df

In [37]:
budapest_point = nearest_point(ds, cities["Budapest"][0], cities["Budapest"][1])
df = daily_aggregation(budapest_point)

In [38]:
print(df.head(10))

       datum        tmax        tmin  tkozep_maxmin  tkozep_t2m   csapadek_mm
0 2006-10-01  296.509338  284.494263     290.501801  290.405701 -9.024809e-09
1 2006-10-02  296.138947  285.307709     290.723328  290.889587  4.577637e-02
2 2006-10-03  297.855621  289.658508     293.757080  294.179932 -3.931024e-09
3 2006-10-04  294.084991  286.512787     290.298889  288.218964  3.067017e+00
4 2006-10-05  291.883026  284.959686     288.421356  287.318237 -1.902598e-09
5 2006-10-06  290.791290  280.276398     285.533844  284.847534 -1.338483e-11
6 2006-10-07  291.457367  279.635254     285.546326  285.808685 -4.280513e-16
7 2006-10-08  291.422852  282.029755     286.726318  286.262329  5.340576e-02
8 2006-10-09  291.429108  278.670624     285.049866  285.901550 -2.466796e-10
9 2006-10-10  292.332672  278.916870     285.624756  285.218933 -1.340083e-09


Látjuk hogy kicsit furák az adatok. A hőmérséklet Kelvinben van megadva, a csapadéknál pedig apró, numerikus zajokat figyelhetünk meg.
Próbaljuk ezt megoldani!

In [39]:
def daily_aggregation(pont_ds):

    daily_max  = pont_ds["mx2t6"].max(dim="step") - kelvin
    daily_min  = pont_ds["mn2t6"].min(dim="step") - kelvin
    daily_mean_maxmin = (daily_max + daily_min) / 2
    daily_mean_t2m    = pont_ds["t2m"].mean(dim="step") - kelvin
    
    # csapadék: kumulatív, a 24h (utolsó) lépés adja a teljes napi összeget
    daily_precipitation = (pont_ds["tp"].isel(step=-1)).clip(min = 0)
    
    df = pd.DataFrame({
            "datum": pont_ds["time"].values,
            "d_ta_maxmin": daily_mean_maxmin.values,
            "d_ta_t2m": daily_mean_t2m.values,
            "d_tx": daily_max.values,
            "d_tn": daily_min.values,
            "csapadek": daily_precipitation.values,
        })
    return df

In [40]:
df2 = daily_aggregation(budapest_point)
print(df2)

        datum  d_ta_maxmin   d_ta_t2m       d_tx       d_tn  csapadek
0  2006-10-01    17.351807  17.255707  23.359344  11.344269  0.000000
1  2006-10-02    17.573334  17.739594  22.988953  12.157715  0.045776
2  2006-10-03    20.607071  21.029938  24.705627  16.508514  0.000000
3  2006-10-04    17.148895  15.068970  20.934998  13.362793  3.067017
4  2006-10-05    15.271362  14.168243  18.733032  11.809692  0.000000
5  2006-10-06    12.383850  11.697540  17.641296   7.126404  0.000000
6  2006-10-07    12.396317  12.658691  18.307373   6.485260  0.000000
7  2006-10-08    13.576309  13.112335  18.272858   8.879761  0.053406
8  2006-10-09    11.899872  12.751556  18.279114   5.520630  0.000000
9  2006-10-10    12.474777  12.068939  19.182678   5.766876  0.000000
10 2006-10-11    11.932541  11.035919  18.664978   5.200104  0.000000
11 2006-10-12    12.838684  11.597595  18.474213   7.203156  0.000000
12 2006-10-13    13.527634  14.220306  18.736877   8.318390  0.000000
13 2006-10-14    13.

Na, máris jófelé tartunk. Viszont találtam a met.hu-n egy leírást ami kicsit árnyalja a helyzetünket.
[met.hu](https://met.hu/eghajlat/magyarorszag_eghajlata/eghajlati_adatsorok/Budapest/leirasok/szamitasok/)

4: MET SZERINTI MÉRÉSI SZABVÁNYOK IMPLEMENTÁLÁSA

A met.hu napi számítási procedúrái kicsit bonyolultabbak. A napi felbontásnál az adott nap értékeinek kiszámításáHoz szükségünk van az azt megelőző és az utána levő nap adataira is. Ez értelemszerűen gondot okoz a hónapok utolsó és első napjainak számításánál mert az adatok hónapos felbontásban vannak. Ezért a grib fájlokat úgy kell megnyitni, hogy két szomszédos hónapot összefésülünk.

A dátum-intervallumunk legelső és legutolsó napját viszont nem tudjuk így lekezelni, ezt el kell fogadnunk.

In [41]:
def get_month_context_range(year, month):
    prev_year, prev_month = (year - 1, 12) if month == 1 else (year, month - 1)
    prev_file = Path(f"tigge_data/{prev_year}-{prev_month:02}_cf.grib")

    next_year, next_month = (year + 1, 1) if month == 12 else (year, month + 1)
    next_file = Path(f"tigge_data/{next_year}-{next_month:02}_cf.grib")

    fname = Path(f"tigge_data/{year}-{month:02}_cf.grib")
    file = xr.open_dataset(fname, engine="cfgrib")

    parts = [file]

    if prev_file.exists():
        prev = xr.open_dataset(prev_file, engine="cfgrib").isel(time=-1)
        parts.insert(0, prev.expand_dims("time"))
    else:
        print(f"[FIGYELEM] Nincs meg az előző hónap fájlja: {prev_file.name}")

    if next_file.exists():
        nxt = xr.open_dataset(next_file, engine="cfgrib").isel(time=0)
        parts.append(nxt.expand_dims("time"))
    else:
        print(f"[FIGYELEM] Nincs meg a következő hónap fájlja: {next_file.name}")

    merged = xr.concat(parts, dim="time")
    return merged.sortby("time")

In [51]:
teszt = get_month_context_range(2008,10)

In [52]:
print(teszt.sizes["time"])

print()
print("Na király.")

33

Na király.


In [44]:
# Napi max és min hőmérséklet

def daily_max_min_methu(pont_ds, day_index):

    prev_day_24h_max = pont_ds["mx2t6"].isel(time = day_index - 1, step = 3)
    current_day_6_12_24_max = pont_ds["mx2t6"].isel(time = day_index, step=[0, 1, 2])
    current_day_max = current_day_6_12_24_max.max()

    daily_max = max(float(prev_day_24h_max), float(current_day_max)) - kelvin


    prev_day_24h_min = pont_ds["mn2t6"].isel(time = day_index - 1, step = 3)
    current_day_6_12_24_min = pont_ds["mn2t6"].isel(time=day_index, step = [0, 1, 2])
    current_day_min = current_day_6_12_24_min.min()

    daily_min = min(float(prev_day_24h_min), float(current_day_min)) - kelvin

    return [daily_max, daily_min]

In [45]:
print(daily_max_min_methu(budapest_point,1))

[22.988946533203148, 12.157708740234398]


In [46]:
# Napi csapadék
def daily_precipitation_methu(pont_ds, day_index):

    current_step6 = float(pont_ds["tp"].isel(time = day_index, step = 0))
    current_step24 = float(pont_ds["tp"].isel(time = day_index, step = 3))
    next_step6 = float(pont_ds["tp"].isel(time = day_index + 1, step = 0))

    precipitation_m = (current_step24 - current_step6) + next_step6

    return max(precipitation_m, 0)


In [47]:
print(daily_precipitation_methu(budapest_point, 8))

1.205731377617897e-09


In [66]:
def daily_aggregation_methu(pont_ds, year, month):
    n_days = pont_ds.sizes["time"]
    
    rows = []
    for day_index in range(n_days):
        datum = pont_ds["time"].isel(time=day_index).values
        
        if day_index - 1 < 0 or day_index + 1 >= n_days:
            nap = pont_ds.isel(time=day_index)
            d_tx = float(nap["mx2t6"].max(dim="step")) - kelvin
            d_tn = float(nap["mn2t6"].min(dim="step")) - kelvin
            csapadek = max(float(nap["tp"].isel(step=-1)), 0)
            
        else:
            d_tx, d_tn = daily_max_min_methu(pont_ds, day_index)
            csapadek = daily_precipitation_methu(pont_ds, day_index)
        
        d_ta = (d_tx + d_tn) / 2 if not (np.isnan(d_tx) or np.isnan(d_tn)) else np.nan
        
        rows.append({
            "datum": pd.Timestamp(datum),
            "d_ta": d_ta,
            "d_tx": d_tx,
            "d_tn": d_tn,
            "d_rs": csapadek,
        })
    
    df = pd.DataFrame(rows)
    
    # csak az adott hónap saját napjai maradjanak - a becsatolt szomszéd-napokat dobjuk el
    df = df[(df["datum"].dt.year == year) & (df["datum"].dt.month == month)].reset_index(drop=True)
    
    return df

In [68]:
budapest_point = nearest_point(teszt, cities["Budapest"][0], cities["Budapest"][1])

df3 = daily_aggregation_methu(budapest_point, 2006, 10)
print(df3)

        datum       d_ta       d_tx       d_tn          d_rs
0  2006-10-01  17.351801  23.359338  11.344263  0.000000e+00
1  2006-10-02  17.573328  22.988947  12.157709  4.577637e-02
2  2006-10-03  20.607065  24.705621  16.508508  2.872467e+00
3  2006-10-04  18.438379  22.455621  14.421136  1.945496e-01
4  2006-10-05  15.632883  18.733026  12.532739  0.000000e+00
5  2006-10-06  13.070032  17.641290   8.498773  0.000000e+00
6  2006-10-07  12.396310  18.307367   6.485254  5.149841e-02
7  2006-10-08  13.579370  18.272852   8.885889  1.907347e-03
8  2006-10-09  11.899866  18.279108   5.520624  1.205731e-09
9  2006-10-10  12.474771  19.182672   5.766870  0.000000e+00
10 2006-10-11  11.932535  18.664972   5.200098  0.000000e+00
11 2006-10-12  12.838678  18.474207   7.203149  0.000000e+00
12 2006-10-13  13.527628  18.736871   8.318384  1.144409e-02
13 2006-10-14  13.283716  18.267389   8.300043  3.433227e-02
14 2006-10-15  11.379480  16.920496   5.838464  3.242492e-02
15 2006-10-16  10.063058

Na akkor ott tartunk, hogy egy város adott év adott hónapjának le tudjuk szedni az adatait, amit akarunk. Nézzük meg, mindegyik várost akkor.

In [70]:
for city in cities:

    lat = cities[city][0]
    lon = cities[city][1]

    pt = nearest_point(teszt, lat, lon)

    df = daily_aggregation_methu(pt, 2006, 10)

    print(f"City: {city}")
    print(df.head(3))
    print('-------')

City: Budapest
       datum       d_ta       d_tx       d_tn      d_rs
0 2006-10-01  17.351801  23.359338  11.344263  0.000000
1 2006-10-02  17.573328  22.988947  12.157709  0.045776
2 2006-10-03  20.607065  24.705621  16.508508  2.872467
-------
City: Debrecen
       datum       d_ta       d_tx       d_tn      d_rs
0 2006-10-01  17.226801  23.830042  10.623560  0.026703
1 2006-10-02  19.051843  24.156915  13.946771  0.310898
2 2006-10-03  21.437143  25.113824  17.760461  0.001907
-------
City: Pécs
       datum       d_ta       d_tx       d_tn      d_rs
0 2006-10-01  18.140848  23.830042  12.451654  0.000000
1 2006-10-02  19.825250  24.160822  15.489679  0.005722
2 2006-10-03  21.757074  26.076715  17.437433  0.000000
-------
City: Szeged
       datum       d_ta       d_tx       d_tn      d_rs
0 2006-10-01  18.545160  25.189417  11.900903  0.000000
1 2006-10-02  19.049387  24.484033  13.614740  0.106812
2 2006-10-03  21.896722  26.549371  17.244073  0.434875
-------
City: Szombathely


Na tök jó fele haladunk. Innen tovább:

-> Ne csak egy hónapra szóljanak a df-ek

-> CSV fájlá alakítás tanulmányozása

-> Egész adatkinyerés automatizációja

In [72]:
teszt2 = get_month_context_range(2014,9)

In [74]:
for city in cities:

    lat = cities[city][0]
    lon = cities[city][1]

    pt = nearest_point(teszt2, lat, lon)

    df = daily_aggregation_methu(pt, 2014, 9)

    print(f"City: {city}")
    print(df.head(3))
    print('-------')

City: Budapest
       datum       d_ta       d_tx       d_tn      d_rs
0 2014-09-01  19.858835  23.695978  16.021692  3.885269
1 2014-09-02  20.132745  24.747614  15.517877  0.007629
2 2014-09-03  22.109796  25.614832  18.604761  0.000000
-------
City: Debrecen
       datum       d_ta       d_tx       d_tn      d_rs
0 2014-09-01  21.121484  26.861993  15.380975  0.320435
1 2014-09-02  22.066339  27.425348  16.707330  0.118256
2 2014-09-03  23.112726  27.110925  19.114526  0.011444
-------
City: Pécs
       datum       d_ta       d_tx       d_tn       d_rs
0 2014-09-01  20.939844  25.563165  16.316522  15.625000
1 2014-09-02  17.589777  20.437067  14.742487   1.934052
2 2014-09-03  19.810968  23.286707  16.335229   7.732391
-------
City: Szeged
       datum       d_ta       d_tx       d_tn      d_rs
0 2014-09-01  22.535547  26.369806  18.701288  1.911163
1 2014-09-02  21.437433  25.939020  16.935846  0.148773
2 2014-09-03  23.041437  27.630457  18.452417  1.544952
-------
City: Szombath

5: TXT-be írás PRÓBÁLKOZÁS

A met.hu-ról nem csv fájlba, hanem txt formátumba lehet letölteni az anyagot ezért énis ezt fogom csinálni

In [ ]:
TXT_DIRECTORY = Path("out_txt") # output mappa létrehozása
TXT_DIRECTORY.mkdir(exist_ok=True)

In [ ]:
print(df.head(5))  # a df amin tudunk kisérletezni

       datum       d_ta       d_tx       d_tn       d_rs
0 2014-09-01  15.720697  19.046747  12.394647  34.448624
1 2014-09-02  14.882501  15.564020  14.200983   9.159088
2 2014-09-03  17.815851  20.956628  14.675073   7.179260
3 2014-09-04  20.000055  24.838190  15.161920   1.358032
4 2014-09-05  19.833963  23.697656  15.970270   1.729965


In [81]:
for i in df:
    print(i)

datum
d_ta
d_tx
d_tn
d_rs


In [ ]:

with open("out_txt/proba.txt", "w") as f:
    f.write("#datum;d_ta;d_tx;d_tn;d_rs")
    f.close()